# Data Vortex — Phase 2: SQL Challenge 6
## Day-of-Week Posting Cadence

### 1. Challenge Description
Analyze posting activity by day of the week using the `timestamp` column in the `posts` table.

Calculates:
- `day_of_week` (Monday through Sunday)
- `day_number` (Monday=1 through Sunday=7)
- `post_count`
- `percentage_of_posts`
- Highest and lowest posting days and difference.

In [ ]:
import os
import sqlite3
import pandas as pd

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_06_day_of_week_cadence.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. Day-of-Week Cadence Query (Monday → Sunday)
Aggregates total posts and computes percentage share per day of the week, ordered from Monday (1) to Sunday (7).

In [ ]:
q_dow = """
WITH daily_posts AS (
    SELECT 
        post_id,
        CAST(strftime('%u', timestamp) AS INTEGER) AS day_number,
        CASE CAST(strftime('%u', timestamp) AS INTEGER)
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
            WHEN 7 THEN 'Sunday'
        END AS day_of_week
    FROM posts
)
SELECT 
    day_of_week,
    day_number,
    COUNT(post_id) AS post_count,
    ROUND(100.0 * COUNT(post_id) / (SELECT COUNT(*) FROM posts), 2) AS percentage_of_posts
FROM daily_posts
GROUP BY day_number, day_of_week
ORDER BY day_number ASC;
"""

df_dow = pd.read_sql_query(q_dow, conn)
df_dow

### 3. Extremes & Variance Query
Identifies highest/lowest posting days and calculates absolute and percentage difference.

In [ ]:
q_extremes = """
WITH daily_posts AS (
    SELECT 
        CAST(strftime('%u', timestamp) AS INTEGER) AS day_number,
        CASE CAST(strftime('%u', timestamp) AS INTEGER)
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
            WHEN 7 THEN 'Sunday'
        END AS day_of_week,
        COUNT(*) AS post_count
    FROM posts
    GROUP BY strftime('%u', timestamp)
),
extremes AS (
    SELECT 
        (SELECT day_of_week FROM daily_posts ORDER BY post_count DESC LIMIT 1) AS highest_day,
        (SELECT post_count FROM daily_posts ORDER BY post_count DESC LIMIT 1) AS highest_count,
        (SELECT day_of_week FROM daily_posts ORDER BY post_count ASC LIMIT 1) AS lowest_day,
        (SELECT post_count FROM daily_posts ORDER BY post_count ASC LIMIT 1) AS lowest_count
)
SELECT 
    highest_day,
    highest_count,
    lowest_day,
    lowest_count,
    highest_count - lowest_count AS absolute_difference,
    ROUND(100.0 * (highest_count - lowest_count) / lowest_count, 2) AS relative_difference_pct
FROM extremes;
"""

df_extremes = pd.read_sql_query(q_extremes, conn)
df_extremes

### 4. Validation Checks
Verifies data integrity across 5 core assertions.

In [ ]:
# Validation 1: Exactly 7 days represented
num_days = len(df_dow)
print(f"1. Days represented:    {num_days} (Expected: 7) -> {'PASS' if num_days == 7 else 'FAIL'}")

# Validation 2: Sum of post counts is 12,000
sum_posts = df_dow['post_count'].sum()
print(f"2. Total post count:    {sum_posts} (Expected: 12000) -> {'PASS' if sum_posts == 12000 else 'FAIL'}")

# Validation 3: Percentages sum to ~100%
sum_pct = round(df_dow['percentage_of_posts'].sum(), 2)
print(f"3. Sum of percentages:  {sum_pct}% (Expected: ~100%) -> {'PASS' if abs(sum_pct - 100.0) < 0.1 else 'FAIL'}")

# Validation 4: Correct ordering (Monday=1 to Sunday=7)
is_ordered = list(df_dow['day_number']) == list(range(1, 8))
print(f"4. Chronological order: {is_ordered} (Expected: True) -> {'PASS' if is_ordered else 'FAIL'}")

# Validation 5: Database unchanged
db_posts = conn.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
print(f"5. Database post count: {db_posts} (Expected: 12000) -> {'PASS' if db_posts == 12000 else 'FAIL'}")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly.")